# Full 1,080-Slab Rigid Validation

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
from zipfile import ZipFile
import hashlib
import json
import os
import queue
import subprocess
import sys
import threading
import time

ROOT = Path("/content/drive/MyDrive/NeuroML_DLIR")
REPO = Path("/content/dlir-partial-fov-main")
DATA = Path("/content/dlir_data")
BUNDLE = ROOT / "rigid_stage_full_validation_bundle.zip"
REFERENCE = ROOT / "outputs/rigid_dev_20261021"
OUT = REFERENCE / "full_validation_1080"
MANIFEST = OUT / "manifest.json"
SCREEN = OUT / "similarity_screen"
REFINE = OUT / "header_refinement"
REPORT = OUT / "paired_report"
EXPECTED_CASES = 1080
OUT.mkdir(parents=True, exist_ok=True)
assert BUNDLE.is_file() or (REPO / "scripts/dl/rigid_refine.py").is_file(), "Rigid-stage source or bundle missing"


def run_live(command, *, cwd=None, label=None, interval=20):
    name = label or Path(str(command[0])).name
    started = time.monotonic()
    print(f"[{datetime.now():%H:%M:%S}] Starting {name}", flush=True)
    environment = dict(os.environ, PYTHONUNBUFFERED="1")
    process = subprocess.Popen(command, cwd=cwd, env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                               errors="replace", bufsize=1)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout:
                messages.put(line)
        finally:
            messages.put(None)
    threading.Thread(target=reader, daemon=True).start()
    try:
        while True:
            try:
                item = messages.get(timeout=interval)
            except queue.Empty:
                print(f"[{datetime.now():%H:%M:%S}] {name}: {time.monotonic()-started:.0f}s elapsed", flush=True)
                continue
            if item is None:
                break
            print(item, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    print(f"[{datetime.now():%H:%M:%S}] Completed {name} in {time.monotonic()-started:.1f}s", flush=True)


## Environment

In [ ]:
run_live([sys.executable, "-m", "pip", "install", "-q", "nibabel", "scipy", "PyYAML", "wandb", "pytest"], label="Dependencies")
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before proceeding"
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))

In [ ]:
archives = sorted(ROOT.glob("dlir-partial-fov*.zip"))
if not (REPO / "scripts/slabsim/evaluation.py").is_file():
    assert archives, "Original source repository archive not found"
    with ZipFile(archives[0]) as archive:
        assert "dlir-partial-fov-main/scripts/slabsim/evaluation.py" in archive.namelist()
        archive.extractall("/content")
assert (REPO / "scripts/slabsim/evaluation.py").is_file()

if not (DATA / "test").is_dir():
    archive = ROOT / "dlir_data.tar.gz"
    assert archive.is_file(), archive
    run_live(["tar", "-xzf", str(archive), "-C", "/content"], label="Restore HCP archive")

validation = sorted((DATA / "test").glob("*/t*/params.json"))
assert len(validation) == EXPECTED_CASES, f"Expected {EXPECTED_CASES} validation slabs, found {len(validation)}"
if not all(all(k in json.loads(v.read_text()) for k in ("pe_axis", "pe_sign", "trt")) for v in validation):
    patch = ROOT / "dlir_data_params_patch.tar.gz"
    assert patch.is_file(), patch
    run_live(["tar", "-xzf", str(patch), "-C", "/content"], label="Restore parameter metadata")
assert all(all(k in json.loads(v.read_text()) for k in ("pe_axis", "pe_sign", "trt")) for v in validation)
print("Validation slabs:", len(validation))

In [ ]:
if not (REPO / "scripts/dl/rigid_refine.py").is_file():
    assert BUNDLE.is_file(), BUNDLE
    with ZipFile(BUNDLE) as archive:
        members = [f for f in archive.namelist() if f.startswith(("scripts/", "tests/", "configs/", "docs/"))]
        assert members
        archive.extractall(REPO, members=members)
print("Rigid-stage source available; simulator and historical training scripts unchanged")
run_live([sys.executable, "-m", "pytest", "tests/test_rigid_dev.py", "tests/test_rigid_refine.py", "-q"],
         cwd=REPO, label="Rigid-stage tests")

## Locked validation

In [ ]:
import wandb
assert wandb.login(), "W&B online authentication required"
sys.path.insert(0, str(REPO / "scripts"))
from dl.rigid_dev_common import available_cases, ensure_manifest
from dl.rigid_refine import RefinementConfig
import yaml

all_cases = available_cases(DATA)
assert len(all_cases) == EXPECTED_CASES
cases = ensure_manifest(DATA, MANIFEST, count=EXPECTED_CASES)
ids = [c["relative_dir"] for c in cases]
assert len(cases) == EXPECTED_CASES and len(set(ids)) == EXPECTED_CASES
assert set(ids) == {c["relative_dir"] for c in all_cases}
assert len({c["subject"] for c in cases}) == 30

pilot = json.loads((REFERENCE / "cnn_similarity_screen/summary.json").read_text())
assert pilot["mode"] == "score" and pilot["oracle_preferred_pct"] >= 75
assert set(pilot["case_ids"]).issubset(set(ids))
config_file = REPO / "configs/rigid_refine.yaml"
config = RefinementConfig(**yaml.safe_load(config_file.read_text())["refinement"])
assert pilot["similarity"] == config.similarity

files = ([BUNDLE] if BUNDLE.is_file() else []) + [config_file, REPO / "scripts/dl/rigid_refine.py",
         REPO / "scripts/dl/eval_rigid_hybrid_dev.py", MANIFEST]
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()
provenance = {str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p.relative_to(REPO)): sha256(p) for p in files}
provenance_path = OUT / "source_checksums.json"
if provenance_path.is_file():
    assert json.loads(provenance_path.read_text()) == provenance, "Experiment inputs changed; do not mix runs"
else:
    provenance_path.write_text(json.dumps(provenance, indent=2) + "\n")
print("Locked validation:", len(cases), "slabs,", len(set(c["subject"] for c in cases)), "subjects")
print("Frozen objective:", config.similarity, "| prior 36-case preference:", pilot["oracle_preferred_pct"], "%")

## Screening and refinement

In [ ]:
score_summary = SCREEN / "summary.json"
if score_summary.is_file():
    result = json.loads(score_summary.read_text())
    assert result.get("case_ids") == ids and result.get("mode") == "score"
    assert result["initial_header"]["attempted"] == EXPECTED_CASES
    print("Complete 1,080-case similarity screening found; reusing results")
else:
    run_live([
        sys.executable, "-u", "scripts/dl/eval_rigid_hybrid_dev.py",
        "--data", str(DATA), "--manifest", str(MANIFEST),
        "--output", str(SCREEN), "--max-cases", str(EXPECTED_CASES),
        "--mode", "score",
    ], cwd=REPO, label="Full validation similarity screening")
result = json.loads(score_summary.read_text())
assert result["case_ids"] == ids
print("Oracle preferred:", result["oracle_preferred_pct"], "%")
print("Header median TRE:", result["initial_header"]["median_tre_mm"], "mm")

In [ ]:
refine_summary = REFINE / "summary.json"
if refine_summary.is_file():
    result = json.loads(refine_summary.read_text())
    assert result.get("case_ids") == ids and result.get("mode") == "refine"
    assert result["refined_header"]["attempted"] == EXPECTED_CASES
    print("Complete 1,080-case refinement found; reusing results")
else:
    run_live([
        sys.executable, "-u", "scripts/dl/eval_rigid_hybrid_dev.py",
        "--data", str(DATA), "--manifest", str(MANIFEST),
        "--output", str(REFINE), "--max-cases", str(EXPECTED_CASES),
        "--mode", "refine", "--score-summary", str(score_summary),
    ], cwd=REPO, label="Full validation header-initialized refinement")
result = json.loads(refine_summary.read_text())
assert result["case_ids"] == ids and result["refined_header"]["attempted"] == EXPECTED_CASES
print("Refined header:", json.dumps(result["refined_header"], indent=2))

## Results

In [ ]:
run_live([
    sys.executable, "-u", "scripts/dl/summarize_rigid_dev.py",
    "--manifest", str(MANIFEST),
    "--inputs", str(SCREEN / "cases.jsonl"), str(REFINE / "cases.jsonl"),
    "--output", str(REPORT),
], cwd=REPO, label="Paired validation summary")
report = json.loads((REPORT / "summary.json").read_text())
for method in ("initial_header", "refined_header"):
    metrics = report[method]
    print(f"{method}: TRE={metrics['median_tre_mm']:.3f} mm; "
          f"rotation={metrics['median_rotation_error_deg']:.3f}°; "
          f"translation={metrics['median_centroid_translation_error_mm']:.3f} mm; "
          f"failure={metrics['failure_pct']:.2f}%; "
          f"TRE<2mm={metrics['tre_below_2mm_pct']:.2f}%; "
          f"runtime={metrics['median_runtime_s']:.3f}s")
print("Saved to", REPORT)

In [ ]:
import csv
import numpy as np
from collections import defaultdict
from dl.rigid_dev_common import load_rows

rows = load_rows(REFINE / "cases.jsonl")
case_by_id = {c["relative_dir"]: c for c in cases}
completed = []
for (case_id, method), row in rows.items():
    if method != "refined_header":
        continue
    info = case_by_id[case_id]
    completed.append({**row, **{k: info.get(k) for k in ("orientation", "thickness_mm", "position", "misalignment")}})

ok = [r for r in completed if r["status"] == "ok"]
failures = [r for r in completed if r["status"] != "ok" or r["failure"]]
worst = sorted(ok, key=lambda r: r["mean_tre_mm"], reverse=True)[:20]
print("Completed cases:", len(completed), "| Software errors:", sum(r["status"] != "ok" for r in completed))
print("Worst cases (TRE mm):")
for row in worst[:10]:
    print(f"{row['mean_tre_mm']:7.3f}  {row['case']}  failure={row['failure']}")
print("TRE p90:", round(float(np.percentile([r['mean_tre_mm'] for r in ok],90)),3), "mm")
print("TRE p95:", round(float(np.percentile([r['mean_tre_mm'] for r in ok],95)),3), "mm")

outlier_path = REPORT / "worst_20.csv"
with outlier_path.open("w", newline="") as f:
    fields = ["case", "subject", "mean_tre_mm", "rot_deg", "trans_mm", "failure", "runtime_s", "orientation", "thickness_mm", "position", "misalignment"]
    writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(worst)

for field in ("orientation", "thickness_mm", "misalignment"):
    groups = defaultdict(list)
    for row in completed:
        groups[str(row.get(field))].append(row)
    print("\n", field)
    for key, group in sorted(groups.items()):
        val = [r["mean_tre_mm"] for r in group if r["status"] == "ok"]
        print(key, "n=", len(group), "median TRE=", round(float(np.median(val)),3) if val else None,
              "failure=", round(100*sum(bool(r["failure"]) for r in group)/len(group),2), "%")


In [ ]:
import matplotlib.pyplot as plt
from dl.rigid_dev_common import load_rows
header_rows = load_rows(SCREEN / "cases.jsonl")
refine_rows = load_rows(REFINE / "cases.jsonl")
paired = [(header_rows.get((case_id,"initial_header")), refine_rows.get((case_id,"refined_header"))) for case_id in ids]
paired = [(h,r) for h,r in paired if h is not None and r is not None and h["status"]=="ok" and r["status"]=="ok"]
assert len(paired) > 0
x = np.array([h["mean_tre_mm"] for h,r in paired]); y = np.array([r["mean_tre_mm"] for h,r in paired])
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(x,y,s=9,alpha=.6)
limit = max(np.quantile(np.r_[x,y], .99),2)
ax.plot([0,limit],[0,limit],linestyle="--",linewidth=1)
ax.set(xlabel="Header TRE (mm)",ylabel="Refined TRE (mm)",title="Paired rigid registration: 1,080 validation slabs")
fig.tight_layout()
fig.savefig(REPORT / "paired_tre.png",dpi=170)
plt.show()
print("Paired improvement:",round(100*np.mean(y<x),2),"% of cases")

## Provenance

In [ ]:
final = {
    "split": "validation",
    "count": len(ids),
    "subjects": len(set(c["subject"] for c in cases)),
    "objective": config.similarity,
    "method": "6-DOF header-initialized numerical optimization",
    "source_checksums": provenance,
    "torch": torch.__version__,
    "summary": report,
}
(OUT / "validation_record.json").write_text(json.dumps(final, indent=2) + "\n")
print("Saved:", OUT)
print("W&B tracking: required online; completed runs linked in each output folder")